# Day 08 · OOP 基础

**前置知识**: Day01~Day07 全部语法,
重点是函数(def/self 雏形)、列表字典(容器思维)、文件 IO(持久化)。

**本节关键问题**:
- 什么是 **类**?什么是 **实例**?为何需要面向对象?
- **self** 到底是什么?
  为什么实例方法第一个参数必须是它?
- **属性**有哪两类?
  类属性 vs 实例属性如何访问、如何修改?
- **@property** 如何实现封装?
  getter / setter 各做什么?
- **继承**如何让子类复用父类代码?
  **super()** 怎么用?
- **isinstance / issubclass** 在类型检查中有何作用?

# 类定义与 __init__ 构造函数

用 **class** 关键字定义一个类,类是创建对象的"模具"。

调用类名加括号 **Dog()** 即可实例化出一个对象,**type()** 可查看任意对象所属的类。

**__init__** 是构造函数,实例化时自动被调用,负责初始化对象属性。

第一个参数 **self** 代表"当前这个实例",后面可接任意参数。

**self.xxx = ...** 把值绑定到当前对象,成为其实例属性。

In [ ]:
class Dog:
    # __init__: 构造函数,实例化时自动调用
    def __init__(self, name, age):
        self.name = name   # 绑定为自己的实例属性
        self.age = age     # 每个实例各有一份

dog1 = Dog("旺财", 3)
dog2 = Dog("小白", 5)
print(dog1.name, dog2.name, dog1.age, dog2.age)
print(type(dog1))         # <class '__main__.Dog'>


# self 与实例方法

**self** 是**当前实例对象本身**。

调用 **dog.bark()** 时,Python 内部会转成 **Dog.bark(dog)**,所以 self 就是 dog。

理解这一点,就能看懂实例方法为何第一个参数永远是 self。

类中定义的普通函数就叫**实例方法**,第一个参数必须是 self。

方法可以**读取**也可以**修改**对象自身的属性,因此在方法内部凡是访问实例自身的成员,
都要加 **self.** 前缀。

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def bark(self):
        # 实例方法:第一个参数 self 指向当前实例
        print(f"{self.name}: 汪汪!")

    def fetch(self, item):
        # 方法内通过 self. 访问实例自身的属性
        print(f"{self.name} 叼回了 {item}")

dog = Dog("旺财", 3)
dog.bark()                  # 普通:实例.方法()
Dog.bark(dog)               # 等价:类.方法(实例)
dog.fetch("飞盘")


# 类属性 vs 实例属性

- **类属性**: 定义在类体里、方法外;所有实例**共享同一份**。
- **实例属性**: 写在 __init__ 里用 **self.xxx = ...**;每个对象**各自拥有一份**,互不影响。

修改类属性必须**通过类名**: **Dog.count += 1**。

若写成 self.count += 1,Python 会为当前实例新建一个同名实例属性,类属性的值并不会被改变,这是最常见的**易错点**。

In [ ]:
class Dog:
    count = 0                  # 类属性:所有实例共享

    def __init__(self, name):
        self.name = name       # 实例属性:各有一份
        Dog.count += 1         # 通过类名修改类属性

d1 = Dog("旺财")
d2 = Dog("小白")
print(f"共创建了 {Dog.count} 条狗")
print(d1.count, d2.count)      # 都能读到类属性
# self.count += 1 会创建实例属性,不影响类属性


# @property getter 只读属性

**@property** 把一个方法伪装成属性,读取时可加入额外逻辑(校验、格式化、懒计算等),对外只暴露一个干净的属性名。

习惯上把真实值存成 **_name**("受保护"约定),对外用 name 访问。

只写 getter 就是一个**只读属性**,外部无法赋值。

In [ ]:
class Student:
    def __init__(self, name):
        self._name = name         # 内部存储

    @property
    def name(self):
        # getter:读取属性时触发,可加逻辑
        return self._name

stu = Student("小明")
print(stu.name)                    # 像读属性,实际调 getter
# stu.name = "小红"              # 只读,赋值报错


# @property setter 带校验的写入

只用 @property 是只读属性。

再加一个 **@name.setter** 装饰的方法,赋值时就会触发它,可在赋值前做**合法性校验**,
不合法直接抛异常。

这样就实现了"对外像属性、对内带逻辑"的封装。

In [ ]:
class Student:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        # setter:赋值时触发,做合法性校验
        if not value:
            raise ValueError("姓名不能为空")
        self._name = value

stu = Student("小明")
stu.name = "小红"                # 调 setter
print(stu.name)
# stu.name = ""                 # ValueError


# 计算属性: @property 实现 average

**计算属性**不存储任何数据,每次访问都是**实时计算**的最新值。

本例用 @property 把 average"伪装"成属性,内部真实计算平均值。

加 **if not self.scores** 做边界校验,空列表返回 0。

In [ ]:
class Student:
    def __init__(self, name, scores):
        self.name = name
        self.scores = scores

    @property
    def average(self):
        # 计算属性:动态算平均值,不是存的数据
        if not self.scores:
            return 0
        return sum(self.scores) / len(self.scores)

stu = Student("小明", [90, 85, 92])
print(f"平均分: {stu.average:.1f}")    # 89.0


# __str__ 与 __repr__ 魔术方法

**__str__** 是字符串表示方法,决定 **print(obj)** 和 **str(obj)** 返回什么。

定义它之后,打印对象不再显示看不懂的地址,而是输出**友好可读**的自定义文本,便于调试与日志。

**__repr__** 是开发者表示,在交互式环境直接输入对象、或 **repr(obj)** 时调用。

理想情况下返回一段**可被 eval() 还原**的字符串,让开发者能准确知道对象的状态。

同一个类可同时定义两者,各服务于不同场景。

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        # 面向用户:友好可读
        return f"{self.name}({self.age}岁)"

    def __repr__(self):
        # 面向开发者:最好能 eval() 还原
        return "Dog('" + self.name + "', " + str(self.age) + ")"

dog = Dog("旺财", 3)
print(dog)                 # __str__
print(repr(dog))           # __repr__


# 继承: 子类复用父类

子类在**类名后的括号**里写上父类名,即可**继承**父类的全部属性和方法。

子类可以**新增**自己的方法,**保留**父类已有的方法,也可以**重写**它们。

本例 Dog(Animal) 继承了 speak 方法,但还没重写它,所以调用的还是父类的默认实现。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name}: ???")

class Dog(Animal):
    pass                     # 继承后拥有 Animal 的一切

dog = Dog("旺财")
dog.speak()                  # 调的是 Animal 的


# 方法重写与 super()

子类定义一个与父类**同名**的方法,就把它覆盖掉了,这叫**重写(override)**。

本例 Dog、Cat 都重写了 speak,各自发出不同叫声,这就是多态最朴素的样子。

子类的 __init__ 如果需要**复用父类的构造逻辑**,应显式调用 **super().__init__(...)**。

除构造函数外,任意方法内部都可用 **super().方法名(...)**
先取回父类的结果,再在上面**追加**自己的逻辑,实现"站在父类肩膀上"的效果,而不是完全另起炉灶。

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name}: ???")

    def info(self):
        return f"名字:{self.name}"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)     # 复用父类构造
        self.breed = breed         # 扩展新属性

    def speak(self):
        # 同名覆盖父类方法,实现多态
        print(f"{self.name}: 汪汪!")

    def info(self):
        base = super().info()      # 站在父类肩膀上
        return f"{base}, 品种:{self.breed}"

class Cat(Animal):
    def speak(self):
        print(f"{self.name}: 喵!")

Dog("旺财", "金毛").speak()
Cat("小白").speak()
print(Dog("旺财", "金毛").info())


# isinstance / issubclass 类型检查

**isinstance(obj, cls)** 判断一个对象是不是某个类(或其祖先类)的实例。

注意:**父类也算**,所以 Dog 实例同时也是 Animal 的实例,这是向上兼容的体现。

**issubclass(A, B)** 判断 A 是不是 B 的子类。

任何类都被视为自身的子类,所以 issubclass(Dog, Dog) 为 True。

它和 isinstance 分别站在"类"和"对象"两个视角做类型检查。

In [ ]:
class Animal:
    pass

class Dog(Animal):
    pass

dog = Dog()
# isinstance:检查对象是否属于某类(含祖先类)
print(isinstance(dog, Dog))      # True
print(isinstance(dog, Animal))   # True(父类也算)
print(isinstance(dog, str))      # False
# issubclass:检查类之间的子类关系
print(issubclass(Dog, Animal))   # True
print(issubclass(Dog, Dog))      # True(自身也是)
print(issubclass(Animal, Dog))   # False


# 综合案例: Student 类

把本节所有知识串起来,写一个完整的 Student 类:

- 用 **@property** 做封装与校验
- 用 **__str__** 友好输出
- 内部维护一个 scores 列表
- 用计算属性动态返回平均分
- 空列表返回 0 做边界保护

In [ ]:
class Student:
    school = "清华大学"             # 类属性:共享

    def __init__(self, name, scores):
        self._name = name
        self.scores = scores

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not value:
            raise ValueError("姓名不能为空")
        self._name = value

    @property
    def average(self):
        # 计算属性:空列表返回 0 做边界保护
        if not self.scores:
            return 0
        return sum(self.scores) / len(self.scores)

    def __str__(self):
        # 友好输出:学校 + 姓名 + 平均分
        return (
            f"[{self.school}] {self.name} "
            f"平均:{self.average:.1f}"
        )

stu = Student("小明", [90, 85, 92])
print(stu)


# 今日小结

**易错点与要点回顾**:
1. 类是模具,实例才是真实可用对象;class 后跟冒号,所有方法体必须缩进。
2. __init__ 负责初始化,self 是实例本身;凡是要跨方法使用的数据,都必须绑定为 **self.xxx**。
3. 类属性**共享**,实例属性**独享**;修改类属性必须用 **类名.**,误用 self. 只会新建同名实例属性。
4. @property 让方法伪装成属性,读写都可加逻辑;
只写 getter 就是只读属性,再加 **@xxx.setter** 才可写。
5. __str__ 面向用户,__repr__ 面向开发者,两者可共存。
6. 继承让子类直接拥有父类的全部功能,super() 是复用父类逻辑的枢纽,任意方法内都可用它。
7. isinstance 检查对象(含父类是否匹配);
issubclass 检查类之间的子类关系。

# 更多练习

- 当堂练: course/lesson08/in_class/practice01-06.py
- 课后作业: course/lesson08/homework/task01-03.py